# AI 에이전트 상태 및 파일 관리 유틸리티

AI 에이전트의 실행 상태, 메시지 기록, 작업 이력 및 참조 문서를 JSON 파일로 저장하는 기능을 제공합니다.  
또한 AI가 생성한 책의 목차(Outline)를 Markdown 파일로 저장하고 불러오는 공통 유틸리티 함수를 정의합니다.

In [1]:
import os
import json

### save_state() — 에이전트 실행 상태 저장
에이전트의 실행 상태(State)를 JSON 파일로 저장하는 함수입니다.  
대화 메시지, 작업 수행 이력, 참고 자료 등의 정보를 직렬화 가능한 형태로 변환하여 `data/state.json` 파일에 저장합니다.

**매개변수**
| 매개변수 | 타입 | 설명 |
|---|---|---|
| `current_path` | `str` | 프로젝트의 기본 디렉터리 경로 |
| `state` | `dict` | 에이전트의 현재 실행 상태를 담은 딕셔너리 |

**State 주요 필드**
| 필드 | 설명 |
|---|---|
| `messages` | 에이전트와 사용자 간 대화 메시지 목록 |
| `task_history` | 에이전트가 수행한 작업 이력 |
| `references` | 검색 쿼리 및 참고 문서 정보 |
| `references["queries"]` | 참고 자료를 검색할 때 사용한 쿼리 목록 |
| `references["docs"]` | 참고한 LangChain Document 목록 |

**반환값**
- `None`: 별도의 값을 반환하지 않습니다.
- 저장 파일: `{current_path}/data/state.json`

**주요 처리 과정**
1. `data` 디렉터리가 존재하지 않으면 생성합니다.
2. `state`에서 대화 메시지를 가져와 클래스 이름과 메시지 내용으로 변환합니다.
3. 작업 이력을 `to_dict()` 메서드를 사용하여 딕셔너리로 변환합니다.
4. 참고 자료에서 검색 쿼리와 문서 메타데이터를 추출합니다.
5. 변환된 데이터를 `state_dict`에 저장합니다.
6. `json.dump()`를 사용하여 JSON 파일로 저장합니다.

**참고**
- `ensure_ascii=False`는 한글을 유니코드 이스케이프 문자열로 변환하지 않고 그대로 저장합니다.
- 메시지의 전체 객체가 아닌 클래스 이름과 `content`만 저장합니다.
- 참고 문서는 본문(`page_content`)이 아닌 메타데이터(`metadata`)만 저장합니다.

In [2]:
def save_state(current_path, state):
    if not os.path.exists(f"{current_path}/data"):
        os.makedirs(f"{current_path}/data")
    
    state_dict = {}

    messages = [(m.__class__.__name__, m.content) for m in state["messages"]]
    state_dict["messages"] = messages
    state_dict["task_history"] = [task.to_dict() for task in state.get("task_history", [])]

    # references
    references = state.get("references", {"queries": [], "docs": []})
    state_dict["references"] = {
        "queries": references["queries"], 
        "docs": [doc.metadata for doc in references["docs"]]
    }
    
    with open(f"{current_path}/data/state.json", "w", encoding='utf-8') as f:
        json.dump(state_dict, f, indent=4, ensure_ascii=False)

### get_outline() — 보고서 목차 불러오기
프로젝트의 `data/outline.md` 파일에서 작성된 보고서 목차를 읽어오는 함수입니다.  
목차 파일이 존재하지 않는 경우 기본 안내 문구를 반환합니다.

**매개변수**
| 매개변수 | 타입 | 설명 |
|---|---|---|
| `current_path` | `str` | 프로젝트의 기본 디렉터리 경로 |

**반환값**
- `str`: 저장된 보고서 목차 또는 기본 안내 문구

**주요 처리 과정**
1. 기본 목차 문자열을 `'아직 작성된 목차가 없습니다.'`로 초기화합니다.
2. `data/outline.md` 파일이 존재하는지 확인합니다.
3. 파일이 존재하면 UTF-8 인코딩으로 읽습니다.
4. 파일 내용을 `outline` 변수에 저장합니다.
5. 저장된 목차 문자열을 반환합니다.

In [3]:
def get_outline(current_path):
    outline = '아직 작성된 목차가 없습니다.'

    if os.path.exists(f"{current_path}/data/outline.md"):
        with open(f"{current_path}/data/outline.md", "r", encoding='utf-8') as f:
            outline = f.read()
            
    return outline

### save_outline() — 보고서 목차 저장
작성된 보고서 목차를 Markdown 파일로 저장하는 함수입니다.  
전달받은 목차 문자열을 프로젝트의 `data/outline.md` 파일에 저장하여 이후 보고서 작성 및 수정 과정에서 재사용할 수 있도록 합니다.

**매개변수**
| 매개변수 | 타입 | 설명 |
|---|---|---|
| `current_path` | `str` | 프로젝트의 기본 디렉터리 경로 |
| `outline` | `str` | Markdown 형식으로 작성된 보고서 목차 |

**반환값**
- `str`: 저장한 목차 문자열
- 저장 파일: `{current_path}/data/outline.md`

**주요 처리 과정**
1. `data` 디렉터리가 존재하는지 확인합니다.
2. 디렉터리가 존재하지 않으면 생성합니다.
3. `outline.md` 파일을 쓰기 모드로 엽니다.
4. 전달받은 목차 문자열을 UTF-8 인코딩으로 저장합니다.
5. 저장한 목차 문자열을 반환합니다.

In [4]:
def save_outline(current_path, outline):
    if not os.path.exists(f"{current_path}/data"):
        os.makedirs(f"{current_path}/data")
    
    with open(f"{current_path}/data/outline.md", "w", encoding='utf-8') as f:
        f.write(outline)
        
    return outline